# Image filtering

In [ ]:
import matplotlib as mpl
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import numpy as np
import cv2 as cv
# this is only necessary for executing in colab
from google.colab.patches import cv2_imshow
%matplotlib inline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
fp = '/content/drive/MyDrive/Cloud/teaching/ComputerVision/data/'

In [ ]:
img = cv.imread(fp+'gisaengchung.png')
# print some information about the image
print(img.shape)
print(type(img))

So, now we know that our image has a certain size and is stored as a ```numpy``` array of - in this case - three dimensions.

Now, let's next look at the image using OpenCV's built-in ```imshow``` function like so:

In [ ]:
# let's make the image first a bit smaller
scale_percent = 50 # percent of original size
width = int(img.shape[1] * scale_percent / 100)
height = int(img.shape[0] * scale_percent / 100)
dim = (width, height)
# resize image
rimg = cv.resize(img, dim, interpolation = cv.INTER_AREA)

# the first parameter for the imshow function simply gives
# the title string for the window and the second the actual
# image array as numpy data
# -- regular call to opencv's imshow like this:
# cv.imshow('parasite',rimg)

# these two lines are necessary so that control is given back
# to the browser and jupyter when we show the images

# waits for a key indefinitely - blocking processing
# cv.waitKey()
# once key has been pressed destroy all windows
# cv.destroyAllWindows()

# call to replacement within colab like this:
cv2_imshow(rimg)



Let's use ```matplotlib``` to show the picture

In [ ]:
plt.figure(None, figsize=(12, 12))
plt.imshow(rimg)
plt.show()

Urgh. The colors are completely different.

This is because OpenCV stores images in BGR (BLUE - GREEN - RED) format, rather than the usual RGB (RED - GREEN - BLUE) format from other languages. So, if we want to show the image properly with ```matplotlib``` we have to convert the image first.

In [ ]:
mrimg = cv.cvtColor(rimg, cv.COLOR_BGR2RGB)
plt.figure(None, figsize=(12, 12))
plt.imshow(mrimg)
plt.show()

In [ ]:
mrimg = cv.cvtColor(rimg, cv.COLOR_BGR2GRAY)
plt.figure(None, figsize=(12, 12))
plt.imshow(mrimg,cmap='gray')
plt.show()

Now, let's cut out a section of the image and let's display it as a grid of values. Most of the code below is to make sure that the plotting works fine and that we can add the pixel intensity values in a nice way to the plot.

In [ ]:
roi = mrimg[455:470,290:305]
# the extent variable is important for ensuring the right
# kind of plotting area!
extent = (0, roi.shape[1], roi.shape[0], 0)
plt.figure(None, figsize=(14, 6))
plt.subplot(1,2,1)
plt.imshow(roi,cmap='gray',extent=extent)
ax = plt.gca()
# this makes sure we have the correct grid lines
major_ticks = np.arange(0, 16, 1)
ax.set_xticks(major_ticks)
ax.set_yticks(major_ticks)
plt.grid(True)
plt.subplot(1,2,2)
plt.imshow(roi,cmap='gray',extent=extent)
ax = plt.gca()
major_ticks = np.arange(0, 16, 1)
ax.set_xticks(major_ticks)
ax.set_yticks(major_ticks)
# plots pixel intensity values with text color depending on
# the pixel value
for i in range(0,15):
    for j in range(0,15):
        if roi[j,i]>127:
            plt.text(i+0.1,j+0.6,str(roi[j,i]),color='black')
        else:
            plt.text(i+0.1,j+0.6,str(roi[j,i]),color='white')
plt.grid(True)

Next, let's plot the image function F(x,y) as a surface.

In [ ]:
# definition of x and y and meshgrid
x = np.arange(0, mrimg.shape[1], 1)
y = np.arange(0, mrimg.shape[0], 1)
xx, yy = np.meshgrid(x, y)

In [ ]:
# code how to do this in Matplotlib, which does NOT work well within Colab
fig = plt.figure(None, figsize=(10, 10))
ax = fig.gca(projection='3d')
ax.plot_surface(yy,xx,mrimg[yy,xx],cmap='gray')
plt.show()

In [ ]:
# code how to do this with plotly, which works well within Colab
import plotly.graph_objects as go

fig = go.Figure(data=[go.Surface(z=mrimg[yy,xx])])

fig.update_layout(title='Image Surface', autosize=False,
                  width=500, height=500,
                  margin=dict(l=65, r=50, b=65, t=90))

fig.show()

## Arithmetics with images

Let's add something to our function

In [ ]:
plt.figure(None, figsize=(12, 12))
plt.imshow(mrimg+50,cmap='gray')
plt.show()

That result cannot be correct - some parts of the image turn black, when actually the whole image should get brighter. This is because of clipping!

In [ ]:
plt.figure(None, figsize=(12, 12))
plt.imshow(np.clip(mrimg.astype(float)+100,0,255),cmap='gray')
plt.show()

Here's the inverse of the image

In [ ]:
plt.figure(None, figsize=(12, 12))
plt.imshow(255-mrimg,cmap='gray')
plt.show()

# Filtering

Let's apply our simple retina-like filter to a signal

In [ ]:
signal = np.zeros((100,1))
signal[10:13]=10;
signal[50:53]=10;
signal[70:90]=10;

filtered_signal = np.zeros_like(signal)
weights = np.array((-1.0,-1.0,1.0,1.0,1.0,-1.0,-1.0))
plt.figure(None, figsize=(6, 6))
plt.plot(weights,linewidth=5)
for i in range(3,len(signal)-3):
    filtered_signal[i]=np.dot(weights,signal[i-3:i+4])

plt.figure(None, figsize=(14, 6))
plt.subplot(1,2,1)
plt.plot(signal)
plt.subplot(1,2,2)
plt.plot(filtered_signal)
plt.show()

In [ ]:
weights = np.array((-1,-1,1,1,1,-1,-1))
weights = weights/np.linalg.norm(weights, ord=2, keepdims=True)
for i in range(3,len(signal)-3):
    filtered_signal[i]=np.dot(weights,signal[i-3:i+4])

plt.figure(None, figsize=(14, 6))
plt.subplot(1,2,1)
plt.plot(signal)
plt.subplot(1,2,2)
plt.plot(filtered_signal)
plt.show()

## Scanline
Let's see what the image function says for our picture at one scanline.

In [ ]:
cuty = 400
plt.figure(None, figsize=(14, 6))
grid = plt.GridSpec(1, 7, wspace=0.4, hspace=0.3)
plt.subplot(grid[0, 0])
plt.imshow(mrimg,cmap='gray')
plt.plot((0,mrimg.shape[1]-1),(cuty,cuty))
plt.subplot(grid[0, 1:4])
oneline = mrimg[cuty,:]
plt.plot(oneline)
plt.show()

## Filtering one image line
Let's apply our filter to the scanline in the image.

In [ ]:
newoneline = np.zeros_like(oneline)
weights = (-1,-1,1,1,1,-1,-1)
for i in range(3,len(oneline)-3):
    newoneline[i]=np.dot(weights,oneline[i-3:i+4])

In [ ]:
plt.figure(None, figsize=(14, 6))
grid = plt.GridSpec(1, 7, wspace=0.4, hspace=0.3)
plt.subplot(grid[0, 0])
plt.imshow(mrimg,cmap='gray')
plt.plot((0,mrimg.shape[1]-1),(cuty,cuty))
plt.subplot(grid[0, 1:4])
plt.plot(oneline)
plt.subplot(grid[0, 4:])
plt.plot(newoneline)
plt.show()

## Filtering in 2D
Let's take the simple averaging filter and run it across the image.

In [ ]:
weights2d = [
    (1,1,1),
    (1,1,1),
    (1,1,1)
]
weights2d = np.array(weights2d)
weights2d = 1/np.sum(weights2d)*weights2d

In [ ]:
filteredmrimg = np.zeros_like(mrimg)
for i in range(1,mrimg.shape[1]-1):
    for j in range(1,mrimg.shape[0]-1):
        filteredmrimg[j,i]=np.sum(weights2d*mrimg[j-1:j+2,i-1:i+2])

In [ ]:
plt.figure(None, figsize=(14, 6))
plt.subplot(1,2,1)
plt.imshow(mrimg,cmap='gray')
plt.subplot(1,2,2)
plt.imshow(filteredmrimg,cmap='gray')
plt.show()

In [ ]:
# this function will filter an roi using a set of weights
def filter_and_plot(weights,roi):
    # these two lines are important as they ensure correct
    # computations!
    weights = weights.astype(float)
    roi = roi.astype(float)
    # this holds the end result
    filtered = np.zeros_like(roi)
    width = int((weights.shape[1]-1)/2)
    height = int((weights.shape[0]-1)/2)
    # do the filtering
    for i in range(height,roi.shape[1]-height):
        for j in range(width,roi.shape[0]-width):
            filtered[j,i]=np.sum(weights*roi[j-width:j+width+1,i-height:i+height+1])
    # plot the original, the filter, and the filtered image
    plt.figure(None, figsize=(14, 6))
    grid = plt.GridSpec(1, 7, wspace=0.4, hspace=0.3)
    plt.subplot(grid[0, 0:3])
    plt.imshow(roi,cmap='gray')
    plt.subplot(grid[0, 3])
    extent = (0, weights.shape[1], weights.shape[0], 0)
    plt.imshow(weights,cmap='gray',extent=extent)
    plt.axis('off')
    if width<5:
        for i in range(0,len(weights)):
            for j in range(0,len(weights)):
                if weights[i,j]>0.5*np.max(weights):
                    plt.text(j+0.1,i+0.6,'{0:.2f}'.format(weights[i,j]),color='black')
                else:
                    plt.text(j+0.1,i+0.6,'{0:.2f}'.format(weights[i,j]),color='white')
    plt.grid(True)
    plt.subplot(grid[0, 4:])
    plt.imshow(filtered,cmap='gray')
    plt.show()
    return(filtered)

In [ ]:
faceroi = mrimg[190:250,314:374]

## Different filters
Let's try out a few simple filters.

In [ ]:
filter_id = np.array([
    (0,0,0),
    (0,1,0),
    (0,0,0)
])
filter_id = 1/np.sum(filter_id)*filter_id

filter_and_plot(filter_id,faceroi)

filter_left = np.array([
    (0,0,0),
    (1,0,0),
    (0,0,0)
])
filter_left = 1/np.sum(filter_left)*filter_left


filter_and_plot(filter_left,faceroi)

filter_blur = np.array([
    (1,1,1),
    (1,1,1),
    (1,1,1)
])
filter_blur = 1/np.sum(filter_blur)*filter_blur


filter_and_plot(filter_blur,faceroi)


filter_contrast = 2*filter_id - filter_blur
filter_contrast = 1/np.sum(filter_contrast)*filter_contrast


fc = filter_and_plot(filter_contrast,faceroi)

## Larger blur
Let's use a bigger averaging filter and see its effects.

In [ ]:
filter_blur_big = np.zeros((15,15))
filter_blur_big[5:10,5:10]=1
filter_blur_big = 1/np.sum(filter_blur_big)*filter_blur_big
f=filter_and_plot(filter_blur_big,faceroi)

## High-pass
Here's the difference between the blurred and the original iamge. Per definition it contains the high frequencies.

In [ ]:
plt.figure(None, figsize=(14, 6))
plt.subplot(1,2,1)
plt.imshow(faceroi,cmap='gray')
plt.subplot(1,2,2)
diffimg = faceroi-f.astype(float)
plt.imshow(diffimg,cmap='gray')
plt.show()

## Gradual sharpening
Let's add more and more of the difference image to the blurred image

In [ ]:
plt.figure(None, figsize=(14, 6))
plt.subplot(1,3,1)
plt.imshow(f,cmap='gray')
plt.subplot(1,3,2)
f = f.astype(float)
plt.imshow(f+0.3*diffimg,cmap='gray')
plt.subplot(1,3,3)
plt.imshow(f+0.6*diffimg,cmap='gray')
plt.show()

## Derivatives
Now, let's derive our image function using the simple derivative approximation.

In [ ]:
filter_dx = np.array([
    (0,0,0),
    (0,-1,1),
    (0,0,0)
])
fdx=filter_and_plot(filter_dx,faceroi)

filter_dy = np.array([
    (0,0,0),
    (0,-1,0),
    (0,1,0)
])
fdy=filter_and_plot(filter_dy,faceroi)

plt.figure(None, figsize=(6, 6))
hsv = np.zeros((fdx.shape[1],fdx.shape[0],3))
hsv[...,1] = 255
ang = np.arctan2(fdx,fdy)
hsv[...,0] = ang*180/np.pi
mag = np.sqrt(fdx*fdx+fdy*fdy)
hsv[...,2] = (mag-np.min(mag))/(np.max(mag)-np.min(mag))*255
plt.imshow(cv.cvtColor(np.uint8(hsv),cv.COLOR_HSV2RGB_FULL))
plt.show()

## Sobel filter
Let's implement the simple Gaussian derivative filter approximation - the Sobel filters.

In [ ]:
filter_sobelx = np.array([
    (-1,0,1),
    (-2,0,2),
    (-1,0,1)
])
fsx=filter_and_plot(1/8*filter_sobelx,faceroi)

filter_sobely = np.array([
    (-1,-2,-1),
    (0,0,0),
    (1,2,1)
])
fsy=filter_and_plot(1/8*filter_sobely,faceroi)

plt.figure(None, figsize=(6, 6))
hsv = np.zeros((fsx.shape[1],fsx.shape[0],3))
hsv[...,1] = 255
ang = np.arctan2(fsx,fsy)
hsv[...,0] = ang*180/np.pi
mag = np.sqrt(fsx*fsx+fsy*fsy)
hsv[...,2] = (mag-np.min(mag))/(np.max(mag)-np.min(mag))*255
plt.imshow(cv.cvtColor(np.uint8(hsv),cv.COLOR_HSV2RGB_FULL))
plt.show()

In [ ]:
# This function performs filtering without plotting
def filter(weights, image):
    """
    Apply 2D filter to image without plotting.
    
    Parameters:
    weights (numpy.ndarray): The filter kernel
    image (numpy.ndarray): The input image
    
    Returns:
    numpy.ndarray: The filtered image
    """
    weights = weights.astype(float)
    image = image.astype(float)
    
    filtered = np.zeros_like(image)
    width = int((weights.shape[1] - 1) / 2)
    height = int((weights.shape[0] - 1) / 2)
    
    for i in range(height, image.shape[1] - height):
        for j in range(width, image.shape[0] - width):
            filtered[j, i] = np.sum(weights * image[j - width:j + width + 1, i - height:i + height + 1])
            
    return filtered

# Create a 500x500 grayscale test image
test_image = np.zeros((500, 500), dtype=np.uint8)
# Add some patterns to make the filtering visible
cv.rectangle(test_image, (150, 150), (350, 350), 255, -1)
cv.circle(test_image, (250, 250), 50, 0, -1)

# Display the test image
plt.figure(None, figsize=(6, 6))
plt.imshow(test_image, cmap='gray')
plt.title("Test Image for Performance Comparison (500×500 pixels)")
plt.show()

# Define filter sizes for comparison
filter_sizes = [3, 5, 9, 15, 23]
repetitions = 50

# Prepare results containers
custom_times = {size: [] for size in filter_sizes}
opencv_times = {size: [] for size in filter_sizes}

# Run timing test
print(f"\nRunning timing comparison over {repetitions} repetitions for each filter size...")
for size in filter_sizes:
    print(f"Testing {size}×{size} filter...", end="", flush=True)
    
    # Create normalized box filter
    box_filter = np.ones((size, size)) / (size * size)
    
    # Time custom filter
    for rep in range(repetitions):
        start_time = time.time()
        _ = filter(box_filter, test_image)
        end_time = time.time()
        custom_times[size].append((end_time - start_time) * 1000)  # Convert to milliseconds
    
    # Time OpenCV filter2D
    for rep in range(repetitions):
        start_time = time.time()
        _ = cv.filter2D(test_image, -1, box_filter)
        end_time = time.time()
        opencv_times[size].append((end_time - start_time) * 1000)  # Convert to milliseconds
    
    print(f" Done. Average times - Custom: {np.mean(custom_times[size]):.2f}ms, OpenCV: {np.mean(opencv_times[size]):.2f}ms")

# Calculate statistics
custom_means = [np.mean(custom_times[size]) for size in filter_sizes]
custom_stds = [np.std(custom_times[size]) for size in filter_sizes]
opencv_means = [np.mean(opencv_times[size]) for size in filter_sizes]
opencv_stds = [np.std(opencv_times[size]) for size in filter_sizes]

# Create plot comparing timing results
plt.figure(figsize=(14, 10))
x = np.arange(len(filter_sizes))
width = 0.35

# Create bar plot with error bars for standard deviation
plt.bar(x - width/2, custom_means, width, label='사용자 정의 필터 (Custom Filter)', 
        color='#5DA5DA', alpha=0.7, yerr=custom_stds, capsize=5)
plt.bar(x + width/2, opencv_means, width, label='OpenCV filter2D 함수', 
        color='#FAA43A', alpha=0.7, yerr=opencv_stds, capsize=5)

plt.xlabel('필터 크기 (Filter Size)', fontsize=12)
plt.ylabel('처리 시간 (Time in milliseconds)', fontsize=12)
plt.title(f'필터 성능 비교 (Filter Performance Comparison)\n각 크기별 {repetitions}회 반복 실행 평균', fontsize=14)
plt.xticks(x, [f'{size}×{size}' for size in filter_sizes], fontsize=11)
plt.legend(fontsize=11)

# Add text with details about repetitions and performance
for i, size in enumerate(filter_sizes):
    custom_mean = custom_means[i]
    opencv_mean = opencv_means[i]
    speed_ratio = custom_mean / opencv_mean if opencv_mean > 0 else float('inf')
    
    # Add performance ratio and repetition count
    plt.text(i, max(custom_mean, opencv_mean) * 1.1, 
             f'속도 비율: {speed_ratio:.2f}x\n표준편차: ±{custom_stds[i]:.2f} / ±{opencv_stds[i]:.2f}', 
             ha='center', va='bottom', fontsize=10, bbox=dict(facecolor='white', alpha=0.5))

# Add information about the test conditions
plt.figtext(0.5, 0.01, 
            f'테스트 환경: 500×500 픽셀 그레이스케일 이미지, 각 필터 크기당 {repetitions}회 반복 측정\n'
            f'박스 필터 (모든 요소가 1인 필터)를 정규화하여 사용', 
            ha='center', fontsize=10, bbox=dict(facecolor='#F0F0F0', alpha=0.5))

plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to make room for the caption
plt.grid(True, alpha=0.3)
plt.show()

# Print detailed statistics
print("\nDetailed Performance Results:")
print("-" * 60)
print(f"{'Filter Size':<15}{'Custom Mean (ms)':<20}{'OpenCV Mean (ms)':<20}{'Speedup':<10}")
print("-" * 60)

for i, size in enumerate(filter_sizes):
    print(f"{size}×{size:<13}{custom_means[i]:<20.2f}{opencv_means[i]:<20.2f}{opencv_means[i]/custom_means[i]:<10.2f}")

print("-" * 60)
print(f"Each test repeated {repetitions} times on a {test_image.shape[0]}×{test_image.shape[1]} gray-scale image.")

NameError: name 'np' is not defined